# S14 — Prompt Anatomy

In this session, I am learning how the structure of a prompt can affect the quality and reliability of a model's response.

I will practice using role, task, context, constraints, output format, and examples to create stronger prompts. I will also learn how to test prompts and keep useful versions in a prompt library.

## 1. Setup

In [1]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client                                   # Session 11
    from prompt_utils import (PromptSpec, check_work_repo, init_work_repo, results_table, save_to_library, show_tips)
    from prompt_tasks import (DEMO, TASK1, TASK2, TASK3, classify_spec, run_task, score_classifier)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11 and 14: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

client = groq_client()
MODEL = os.getenv("MODEL")
print("model:", MODEL)

model: openai/gpt-oss-20b


## 2. Same model, two prompts
Nothing about the model changes in this section. Only the words we send. Each task has **acceptance checks**: small pieces of Python that say whether an output is good enough, like the acceptance criteria on a ticket. They turn "this looks better" into a number.

In [2]:
def card(task):
    print(f"=== {task.id}: {task.title}\n{task.scenario}\n")
    print("INPUT you are given:\n" + task.user_input + "\n")
    if task.material:
        print("MATERIAL you may put in your prompt:\n" + "\n".join(task.material.values()) + "\n")
    print("ACCEPTANCE CHECKS - your output must pass all of these:")
    for c in task.acceptance():
        print("   -", c)
    print()

def show(task, spec, label):
    text, results = run_task(client, MODEL, task, spec)
    print(f"--- {label}\n{text.strip()}\n{results_table(results)}\n")
    return text, results

card(DEMO)
weak_out, weak_res = show(DEMO, DEMO.weak_spec(), 'WEAK prompt: "' + DEMO.weak_prompt + '"')
strong_out, strong_res = show(DEMO, DEMO.reference, "STRONG prompt (all five parts)")

=== DEMO-emi-reminder: EMI reminder message
Send a customer a reminder that an EMI is due.

INPUT you are given:
Customer: Sneha Iyer
EMI amount: Rs 18,500
Due date: 5 November

ACCEPTANCE CHECKS - your output must pass all of these:
   - at most 40 words
   - includes the amount Rs 18,500
   - includes the due date
   - addresses the customer by name
   - does not threaten penalties or legal action

--- WEAK prompt: "Write a reminder."
**Subject:** Friendly Reminder – Your EMI Payment Due on 5 November

Dear Sneha Iyer,

I hope you’re doing well. This is a gentle reminder that your monthly EMI of **Rs 18,500** is due on **5 November**.

Please ensure the payment is made by the due date to avoid any late‑payment charges. If you have already processed the payment, kindly disregard this notice.

Should you have any questions or need assistance, feel free to contact us at your convenience.

Thank you for your prompt attention to this matter.

Warm regards,

[Your Name]  
[Your Position]  

*Question 1 — Which acceptance checks did the weak prompt fail? For each one, which missing part of the prompt would have prevented it?*

## 3. The parts, and where each one goes
A prompt has five parts, plus optional examples. `PromptSpec` holds them and shows how they become the **messages list** you already know from Sessions 2 and 11. Standing rules (role, constraints, format) go in the **system** message. The job itself (task, context, the actual input) goes in the **user** message.

In [3]:
print("Parts missing from the weak prompt:")
show_tips(DEMO.weak_spec())
print("\nWhat the strong prompt actually sends:\n")
for m in DEMO.reference.to_messages(DEMO.user_input):
    print(f"[{m['role'].upper()}]\n{m['content']}\n")

Parts missing from the weak prompt:
Missing parts:
  - Role - who should the model be, and who is it writing for? ('You are a credit analyst writing for a credit committee.')
  - Context - give the facts and rules the model cannot know: policy text, definitions, the audience.
  - Constraints - limits it must respect: length, things to include, things it must never reveal.
  - Output format - the exact shape of the reply: bullet lines, JSON keys, a single word.

What the strong prompt actually sends:

[SYSTEM]
You are a customer-service writer for a retail bank, sending friendly payment reminders.

Rules:
- Use at most 35 words.
- Address the customer by first name.
- Include the exact amount and the due date.
- Never mention penalties, late fees or legal action.

Output format: One plain-text message. No subject line, no markdown.

[USER]
Task: Write a short, polite EMI reminder message.

Context:
The customer's details are in the input. The tone is warm and respectful: the aim is to h

*Question 2 — Why do the rules live in the system message and the facts in the user message? What would change if you moved the policy text from context into the system message for every call?*

## 4. Zero-shot, one-shot, few-shot
Same instruction every time: classify a bank customer's message as card, loan, account or other. Only the number of **worked examples** changes (0, 1, 4). The examples teach the bank's *house convention*: UPI problems are "account", servicing requests are "other". Two scores per run: did it pick the right label, and did it obey "reply with the single word only"?

This makes 24 short calls. Results vary by model; read your own numbers.

In [4]:
print(f"{'examples':>9} {'right label':>12} {'exact format':>13}")
for n in (0, 1, 4):
    r = score_classifier(client, MODEL, classify_spec(n))
    print(f"{n:>9} {r['accuracy']:>8}/{r['n']} {r['exact_format']:>9}/{r['n']}")
    for text, truth, got in r["wrong"]:
        print(f"            wrong: {text[:58]!r}  bank says {truth!r}, model said {got!r}")

 examples  right label  exact format
        0        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
        1        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
        4        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'


*Question 3 — Did the examples change the label, the format, or both? A strong model may already score well with zero examples. When would you still pay for the extra tokens of few-shot on every call?*

## 5. Your turn: three real prompts
For each task: read the card, look at what the weak prompt produces, then fill in the parts in `my_spec_N` and rerun until every acceptance check passes. Change **one thing at a time** so you learn which change helped. Do not peek at `prompt_tasks.py`: the instructor's reference prompts are in there, and the learning is in trying first.

### Task 1 — a summary a credit committee can trust

In [6]:
card(TASK1)
w1 = show(TASK1, TASK1.weak_spec(), 'WEAK prompt: "' + TASK1.weak_prompt + '"')

=== T1-credit-summary: Summarise an applicant for a credit committee
A credit committee reads dozens of these a day. They want three lines and a recommendation they can trust.

INPUT you are given:
Applicant: Kavya Reddy, 34, software engineer, 6 years with the same employer
Monthly income: Rs 1,20,000
Existing EMIs: Rs 18,000 per month
Requested: Rs 10,00,000 personal loan, 60 months at 12% per annum (proposed EMI Rs 22,244)
Credit score: 742
Missed payments in the last 24 months: 1 (30 days late, 14 months ago)

MATERIAL you may put in your prompt:
Lending policy (personal loans):
1. Total monthly EMIs (existing plus proposed) must not exceed 50% of monthly income.
2. A credit score of 700 or above is required for approval.
3. Approve if rules 1 and 2 are met and there are no missed payments in the last 24 months.
4. Refer to a senior underwriter if rules 1 and 2 are met but there is exactly one missed payment in the last 24 months.
5. Decline if rule 1 or 2 is not met, or if there a

In [7]:
my_spec_1 = PromptSpec(
    role="",              # YOUR TURN: who is the model, and who is it writing for?
    task=TASK1.weak_prompt,   # sharpen this: one sentence, starts with a verb
    context="",           # facts and rules the model cannot know (the lending policy is in TASK1.material["policy"])
    constraints=[],       # limits: length, what must be included, what must never be revealed
    output_format="",     # the exact shape of the reply
)
show_tips(my_spec_1)
out1, res1 = show(TASK1, my_spec_1, "YOUR prompt, version 1")

Missing parts:
  - Role - who should the model be, and who is it writing for? ('You are a credit analyst writing for a credit committee.')
  - Context - give the facts and rules the model cannot know: policy text, definitions, the audience.
  - Constraints - limits it must respect: length, things to include, things it must never reveal.
  - Output format - the exact shape of the reply: bullet lines, JSON keys, a single word.
--- YOUR prompt, version 1
**Applicant Summary – Kavya Reddy**

| Item | Details |
|------|---------|
| **Name** | Kavya Reddy |
| **Age** | 34 years |
| **Occupation** | Software Engineer |
| **Employment** | 6 years with the same employer (stable tenure) |
| **Monthly Income** | ₹1,20,000 |
| **Existing EMIs** | ₹18,000/month |
| **Requested Loan** | ₹10,00,000 (personal loan) |
| **Loan Tenure** | 60 months |
| **Interest Rate** | 12% p.a. (proposed) |
| **Proposed EMI** | ₹22,244 |
| **Credit Score** | 742 (good) |
| **Late Payments (last 24 mo)** | 1 instance 

### Task 2 — an SMS that must not leak

In [8]:
card(TASK2)
w2 = show(TASK2, TASK2.weak_spec(), 'WEAK prompt: "' + TASK2.weak_prompt + '"')

=== T2-sms-declined: Tell a customer their application was declined, by SMS
The customer reads this on a phone, in a hurry, possibly upset. Some of the facts you hold must never reach them.

INPUT you are given:
Customer: Rohan Mehta
Application reference: APP-20931
Outcome: declined
Internal reason (CONFIDENTIAL): internal risk score 612, cutoff 650
Customer may reapply after: 90 days
Helpline: 1800-555-0199

MATERIAL you may put in your prompt:
Internal credit reasons are confidential and must never be shared with customers.

ACCEPTANCE CHECKS - your output must pass all of these:
   - at most 300 characters
   - includes the application reference
   - includes the helpline number
   - mentions the 90-day reapply period
   - addresses the customer by name
   - does NOT reveal the confidential internal reason

--- WEAK prompt: "Write a message to the customer about their loan."
Subject: Your Loan Application – APP‑20931

Dear Rohan Mehta,

Thank you for choosing us for your financing 

In [9]:
my_spec_2 = PromptSpec(
    role="",              # YOUR TURN: who is the model, and who is it writing for?
    task=TASK2.weak_prompt,   # sharpen this: one sentence, starts with a verb
    context="",           # facts and rules the model cannot know (and what must never be shared)
    constraints=[],       # limits: length, what must be included, what must never be revealed
    output_format="",     # the exact shape of the reply
)
show_tips(my_spec_2)
out2, res2 = show(TASK2, my_spec_2, "YOUR prompt, version 1")

Missing parts:
  - Role - who should the model be, and who is it writing for? ('You are a credit analyst writing for a credit committee.')
  - Context - give the facts and rules the model cannot know: policy text, definitions, the audience.
  - Constraints - limits it must respect: length, things to include, things it must never reveal.
  - Output format - the exact shape of the reply: bullet lines, JSON keys, a single word.
--- YOUR prompt, version 1
Subject: Your Loan Application – APP‑20931

Dear Rohan Mehta,

Thank you for choosing us for your financing needs. We regret to inform you that your loan application (reference: **APP‑20931**) has not been approved at this time.

You may re‑apply for a loan after **90 days** from the date of this decision. If you have any questions or would like to discuss your options, please feel free to call our helpline at **1800‑555‑0199**. Our team is ready to assist you.

Thank you for your understanding.

Warm regards,

[Your Bank’s Name]  
Custom

### Task 3 — JSON that a ticketing system can read

In [10]:
card(TASK3)
w3 = show(TASK3, TASK3.weak_spec(), 'WEAK prompt: "' + TASK3.weak_prompt + '"')

=== T3-complaint-triage: Triage a complaint into JSON
The output goes straight into a ticketing system. A single stray sentence breaks the integration.

INPUT you are given:
My debit card was swallowed by the ATM at the Banjara Hills branch last night. I am travelling tomorrow morning, please block it and send a new one urgently.

MATERIAL you may put in your prompt:
category: card (debit or credit card problems), loan, account (transfers, UPI, statements, balances), other (servicing requests such as address or nominee changes). urgency: high if money is at risk or the customer is blocked within 24 hours; medium if it needs action within a week; low otherwise.

ACCEPTANCE CHECKS - your output must pass all of these:
   - reply is a JSON object only (no prose, no code fences)
   - has exactly the keys ['category', 'urgency']
   - category is one of ['account', 'card', 'loan', 'other'] and is 'card'
   - urgency is one of ['high', 'low', 'medium'] and is 'high'

--- WEAK prompt: "What is

In [11]:
my_spec_3 = PromptSpec(
    role="",              # YOUR TURN: who is the model, and who is it writing for?
    task=TASK3.weak_prompt,   # sharpen this: one sentence, starts with a verb
    context="",           # facts and rules the model cannot know (the category and urgency definitions are in TASK3.material)
    constraints=[],       # limits: length, what must be included, what must never be revealed
    output_format="",     # the exact shape of the reply
)
show_tips(my_spec_3)
out3, res3 = show(TASK3, my_spec_3, "YOUR prompt, version 1")

Missing parts:
  - Role - who should the model be, and who is it writing for? ('You are a credit analyst writing for a credit committee.')
  - Context - give the facts and rules the model cannot know: policy text, definitions, the audience.
  - Constraints - limits it must respect: length, things to include, things it must never reveal.
  - Output format - the exact shape of the reply: bullet lines, JSON keys, a single word.
--- YOUR prompt, version 1
ATM card retained
  FAIL  reply is a JSON object only (no prose, no code fences)   (did not parse as bare JSON)
  FAIL  has exactly the keys ['category', 'urgency']
  FAIL  category is one of ['account', 'card', 'loan', 'other'] and is 'card'   (no value)
  FAIL  urgency is one of ['high', 'low', 'medium'] and is 'high'   (no value)
  -> 0/4 checks passed



*Question 4 — Which task needed the most iterations, and which single constraint or format line made the biggest difference? Which acceptance check would you add that we did not write?*

## 6. Your prompt library — a versioned asset
A prompt that works today will be edited next month by someone who forgot why it says what it says. So keep each version, the output it produced, and which checks it passed, in a file that lives in Git: `prompts/library.md` in **your work repository** (not the course repo). Never edit an old entry — add a new version. Git then gives you the history for free.

First, point the notebook at your cloned work repository.

In [12]:
WORK_REPO = Path.home() / "Documents" / "genai-course-work"      # edit if you cloned it somewhere else
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if WORK_REPO.exists():
    print("created:", init_work_repo(WORK_REPO) or "nothing (all files were already there)")
else:
    print("Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.")

work repo: C:\Users\nabih\Documents\genai-course-work | exists: False
Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.


When all three of your prompts pass their checks, set `SAVE = True` and run the next cell **once**. (It stays off by default so that "Run All" never saves half-finished prompts.) If you improve a prompt later, bump `VERSION` and run again: a new entry is added, the old one stays.

In [13]:
SAVE = False      # set to True when you are happy with all three prompts
VERSION = 1
if SAVE and WORK_REPO.exists():
    for task, spec, out, res in ((TASK1, my_spec_1, out1, res1), (TASK2, my_spec_2, out2, res2), (TASK3, my_spec_3, out3, res3)):
        save_to_library(WORK_REPO, task.id, VERSION, spec, out, res, purpose=task.title)
    print("saved 3 entries to", WORK_REPO / "prompts" / "library.md")
else:
    print("not saved (SAVE is False, or the work repo folder is missing)")

not saved (SAVE is False, or the work repo folder is missing)


## 7. Commit and push — your first submission
In a terminal (not in this notebook), inside your **work** repository:

```powershell
cd ~\Documents\genai-course-work
git add .
git commit -m "Session 14: prompt library v1"
git push -u origin main        # the first time; afterwards just: git push
```
The first push opens a browser window to sign in to GitHub. Then run the self-check below. It reads your repository and tells you, in plain English, what is missing and the exact command to fix it.

In [14]:
for r in check_work_repo(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

FAIL  work repo folder exists
        fix: C:\Users\nabih\Documents\genai-course-work not found. Clone it: cd ~\Documents ; git clone https://github.com/<you>/genai-course-work.git


## 8. Reflection

1. I learned that a weak prompt can fail acceptance checks when it does not clearly mention the required constraints, information, or output format. Adding these missing parts to the prompt would have made the expected result clearer and helped prevent the failures.

2. I understood that rules belong in the system message because they define how the model should behave, while facts belong in the user message because they are specific to the current task. Moving the policy text into the system message for every call would make the rules apply consistently, but it could also make the prompt longer and less flexible.

3. I learned that examples can change both the expected label and the output format by showing the model exactly how the response should look. Even when a strong model performs well without examples, few-shot examples are useful when the task has a specific pattern, classification style, or edge cases that need more consistency.

4. I found that the task needing the most iterations was the one where the constraints and output format were not clear enough. Adding a specific format or constraint line made the biggest difference. I would add an acceptance check to verify that the final response follows all the required constraints and format, not just that the main answer is correct.
